# Muziekclassificatie -- Drempelwaarde-model (Supervised)

**Project R.E.M. -- Notebook 3 van 4**

## Doel

Classificeert nummers uit de Spotify-bibliotheek van deelnemers als **CALM**, **ENERGY** of **OTHER**
op basis van een gewogen arousal-score berekend uit Spotify audio-kenmerken.

**Onderzoeksvraag (RQ5, deels):** *Kunnen we elk nummer in de bibliotheek van een deelnemer
automatisch classificeren voor playlistgeneratie, voorbij handmatige BPM/energy-drempels?*

## Aanpak

1. **Voorfilter** -- verwijder live-opnames (liveness > 0.80) en gesproken-woord tracks (speechiness > 0.66)
2. **Normalisatie** -- MinMaxScaler per deelnemer zodat elk kenmerk het bereik [0,1] krijgt
3. **Arousal-score** -- gewogen som van audio-kenmerken:
   ```
   arousal = 0.35 x energy + 0.30 x tempo + 0.20 x loudness - 0.10 x acousticness + 0.05 x danceability
   ```
4. **Classificatie** -- drempelwaarden op de arousal-score + valence-vloer voor CALM

## Waarom per-deelnemer normalisatie?

De muziekbibliotheek van elke deelnemer heeft een uniek audio-featurebereik
(bijv. iemand die alleen elektronische muziek luistert heeft andere min/max tempo dan een klassiekluisteraar).
Per-deelnemer MinMaxScaler zorgt dat de drempelwaarden dezelfde relatieve betekenis hebben
voor elk individu. Zie: [sklearn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

## Gebruik

| Instelling | Betekenis |
|---|---|
| `REUSE_MODEL = False` | Fit nieuwe scaler en sla op (aanbevolen bij nieuwe deelnemers) |
| `REUSE_MODEL = True` | Laad bestaande scaler vanuit `models/music_classification/` |

**Vereiste:** voor elk deelnemer moet `data/playlists/{codename}/playlists_generated/combined.csv`
bestaan (gegenereerd door `uv run python scripts/playlists/spotify_cli.py prepare {codename}`).

In [ ]:
from __future__ import annotations

import json
import math
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
print('Imports OK')

## Configuratie

In [ ]:
# REUSE_MODEL: True  -> laad bestaande scaler (snel)
#              False -> fit nieuwe scaler en sla op
REUSE_MODEL = False

# PARTICIPANT: 'all' -> alle deelnemers automatisch detecteren
#              string -> bijv. 'peer'
PARTICIPANT = 'all'

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paden
PROJECT_ROOT  = Path().resolve().parent.parent
PLAYLISTS_DIR = PROJECT_ROOT / 'data' / 'playlists'
ANALYSIS_DIR  = PROJECT_ROOT / 'data' / 'analysis'
MODELS_DIR    = PROJECT_ROOT / 'models' / 'music_classification'

# Audio-kenmerken die worden gebruikt voor scoring
SCORING_FEATURES = [
    'tempo', 'energy', 'loudness', 'valence',
    'danceability', 'acousticness', 'instrumentalness', 'speechiness',
]

# Gewichten voor arousal-score (positief = verhoogt arousal, negatief = verlaagt)
# Gewichten zijn gebaseerd op muziekpsychologisch onderzoek naar arousaldimensies.
AROUSAL_WEIGHTS = {
    'energy':       0.35,
    'tempo':        0.30,
    'loudness':     0.20,
    'acousticness': -0.10,
    'danceability': 0.05,
}

# Voorfilter-drempels (gebaseerd op Spotify API-documentatie)
SPEECHINESS_MAX = 0.66   # > 0.66 = gesproken woord / podcast
LIVENESS_MAX    = 0.80   # > 0.80 = hoge kans op live-opname

# Classificatiedrempels
CALM_THRESHOLD   = 0.35
ENERGY_THRESHOLD = 0.65
VALENCE_FLOOR    = 0.25  # CALM-nummers moeten positief genoeg zijn (geen droeve muziek)

# Visuele stijl (donker thema, consistent met Shiny-app)
DARK = {
    'figure.facecolor': '#0f1218', 'axes.facecolor': '#181e2a',
    'axes.edgecolor': '#4a5568', 'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#e2e8f0', 'ytick.color': '#e2e8f0',
    'text.color': '#e2e8f0', 'grid.color': '#2d3748', 'grid.alpha': 0.5,
    'font.family': 'monospace', 'legend.facecolor': '#1a2035',
    'legend.edgecolor': '#4a5568',
}
plt.rcParams.update(DARK)
plt.rcParams['figure.dpi'] = 120

# Okabe-Ito kleuren per klasse (kleurenblind-vriendelijk)
CLASS_COLORS = {'calm': '#56b4e9', 'energy': '#e69f00', 'other': '#999999'}
CLASS_ORDER  = ['calm', 'energy', 'other']

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'REUSE_MODEL  : {REUSE_MODEL}')
print(f'PARTICIPANT  : {PARTICIPANT}')

## 1. Data laden

In [ ]:
def load_combined(codename: str) -> pd.DataFrame:
    """Laad combined.csv (alle Spotify-nummers) voor een deelnemer."""
    path = PLAYLISTS_DIR / codename / 'playlists_generated' / 'combined.csv'
    if not path.exists():
        raise FileNotFoundError(
            f"Invoerbestand niet gevonden voor '{codename}'.\n"
            f"Verwacht: {path}\n"
            f"Voer eerst uit: uv run python scripts/playlists/spotify_cli.py prepare {codename}"
        )
    df = pd.read_csv(path)
    df['participant'] = codename
    return df


def detect_participants() -> list[str]:
    """Detecteer alle deelnemers met een combined.csv."""
    return sorted(
        d.name for d in PLAYLISTS_DIR.iterdir()
        if d.is_dir() and (d / 'playlists_generated' / 'combined.csv').exists()
    )


if PARTICIPANT == 'all':
    participants = detect_participants()
    if not participants:
        raise FileNotFoundError(
            f'Geen combined.csv gevonden onder {PLAYLISTS_DIR}.\n'
            'Verwerk eerst de playlists met de spotify_cli.py pipeline.'
        )
else:
    participants = [PARTICIPANT]

frames: dict[str, pd.DataFrame] = {}
for codename in participants:
    df_raw = load_combined(codename)
    frames[codename] = df_raw
    print(f'  {codename}: {len(df_raw)} nummers geladen')

print(f'\nTotaal: {len(participants)} deelnemer(s)')

## 2. Validatie & Voorverwerking

Live-opnames en gesproken-woord tracks worden gefilterd op basis van Spotify's
`liveness` en `speechiness` scores. Deze tracks verstoren de arousal-score
omdat ze andere audio-eigenschappen hebben dan studio-opnames.

In [ ]:
def validate_and_clean(df: pd.DataFrame, codename: str) -> pd.DataFrame:
    """Controleer vereiste kolommen, verwijder duplicaten en NaN-rijen."""
    required = SCORING_FEATURES + ['liveness']
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f'[{codename}] Ontbrekende kolommen: {missing_cols}')

    if 'uri' in df.columns:
        n_dupes = df.duplicated(subset=['uri'], keep='first').sum()
        df = df.drop_duplicates(subset=['uri'], keep='first').copy()
    else:
        n_dupes = df.duplicated(subset=['name', 'artists'], keep='first').sum()
        df = df.drop_duplicates(subset=['name', 'artists'], keep='first').copy()

    n_before = len(df)
    df = df.dropna(subset=required).copy()
    n_nan = n_before - len(df)
    print(f'  {codename}: {len(df)} unieke nummers ({n_dupes} duplicaten, {n_nan} NaN-rijen verwijderd)')
    return df


def prefilter(df: pd.DataFrame, codename: str) -> pd.DataFrame:
    """Verwijder live-opnames (liveness > 0.80) en gesproken-woord (speechiness > 0.66)."""
    speech_mask = df['speechiness'] > SPEECHINESS_MAX
    live_mask   = df['liveness']    > LIVENESS_MAX
    n_removed   = (speech_mask | live_mask).sum()
    df = df[~speech_mask & ~live_mask].copy()
    if n_removed:
        print(f'  {codename}: {n_removed} tracks gefilterd ({speech_mask.sum()} speech, {live_mask.sum()} live)')
    return df


cleaned: dict[str, pd.DataFrame] = {}
for codename, df_raw in frames.items():
    df = validate_and_clean(df_raw, codename)
    df = prefilter(df, codename)
    cleaned[codename] = df

print('\nVoorverwerking voltooid.')

## 3. Normalisatie & Arousal-score

**Waarom MinMaxScaler?**
Elke genormaliseerde waarde wordt een percentage van het bereik van dat kenmerk in de bibliotheek
van die specifieke deelnemer. De gewichten betekenen dan letterlijk:
'35% van het volledige energy-bereik van deze persoon draagt bij aan de score.'

**Arousal-scoreformule (op genormaliseerde [0,1]-kenmerken):**
```
arousal = 0.35 x energy + 0.30 x tempo + 0.20 x loudness - 0.10 x acousticness + 0.05 x danceability
```

**Classificatieregels:**
- `arousal < 0.35` EN `valence >= 0.25` -> **CALM** (laag arousal, niet te droef)
- `arousal > 0.65` -> **ENERGY** (hoog arousal)
- Al het overige -> **OTHER** (middengebied, opvangnet)

De valence-vloer voor CALM voorkomt dat droeve, rustige muziek als CALM wordt geclassificeerd --
voor stressreductie is positive valence wenselijk.

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)


def fit_or_load_scaler(codename: str, X: np.ndarray) -> MinMaxScaler:
    """Fit een nieuwe MinMaxScaler of laad een bestaande."""
    scaler_path = MODELS_DIR / f'{codename}_scaler.pkl'
    if REUSE_MODEL:
        if not scaler_path.exists():
            raise FileNotFoundError(
                f'REUSE_MODEL=True maar scaler niet gevonden: {scaler_path}\n'
                'Zet REUSE_MODEL=False en voer het notebook opnieuw uit.'
            )
        scaler = joblib.load(scaler_path)
        print(f'  {codename}: scaler geladen vanuit {scaler_path.name}')
    else:
        scaler = MinMaxScaler()
        scaler.fit(X)
        joblib.dump(scaler, scaler_path)
        print(f'  {codename}: scaler gefittet en opgeslagen -> {scaler_path.name}')
    return scaler


def classify_pipeline(
    df: pd.DataFrame,
    codename: str,
    calm_threshold: float = CALM_THRESHOLD,
    energy_threshold: float = ENERGY_THRESHOLD,
    valence_floor: float = VALENCE_FLOOR,
) -> tuple:
    """
    Volledige classificatiepipeline voor een deelnemer.
    Retourneert (df_met_klassen, df_genormaliseerd, scaler).
    """
    X = df[SCORING_FEATURES].values
    scaler = fit_or_load_scaler(codename, X)

    X_scaled  = scaler.transform(X)
    df_scaled = pd.DataFrame(X_scaled, columns=SCORING_FEATURES, index=df.index)

    df = df.copy()
    df['arousal_score'] = sum(
        weight * df_scaled[feat] for feat, weight in AROUSAL_WEIGHTS.items()
    )

    def classify(arousal: float, valence_norm: float) -> str:
        if arousal < calm_threshold and valence_norm >= valence_floor:
            return 'calm'
        elif arousal > energy_threshold:
            return 'energy'
        return 'other'

    df['class'] = [
        classify(a, v)
        for a, v in zip(df['arousal_score'], df_scaled['valence'])
    ]

    return df, df_scaled, scaler


results: dict = {}
for codename, df in cleaned.items():
    df_classified, df_scaled, scaler = classify_pipeline(df, codename)
    results[codename] = (df_classified, df_scaled, scaler)

print('\nClassificatie voltooid.')

## 4. Opslaan

Sla geclassificeerde nummers en config op. De Shiny-app en playlist-generator lezen `classified_songs.csv`.

In [ ]:
# classified_songs.csv per deelnemer
internal_cols = {'kmeans_cluster', 'kmeans_label', 'gmm_cluster', 'gmm_label', 'gmm_confidence'}

for codename, (df_c, _, _) in results.items():
    out_dir = ANALYSIS_DIR / codename
    out_dir.mkdir(parents=True, exist_ok=True)
    save_cols = [c for c in df_c.columns if c not in internal_cols]
    csv_path = out_dir / 'classified_songs.csv'
    df_c[save_cols].to_csv(csv_path, index=False)
    print(f'  {codename}: {len(df_c)} nummers -> {csv_path.relative_to(PROJECT_ROOT)}')

# Config per deelnemer (drempelwaarden + klasseverdeling)
for codename, (df_c, _, _) in results.items():
    counts = df_c['class'].value_counts()
    config = {
        'created':            datetime.now().isoformat(),
        'participant':        codename,
        'n_songs_total':      len(frames[codename]),
        'n_songs_classified': len(df_c),
        'scoring_features':   SCORING_FEATURES,
        'arousal_weights':    AROUSAL_WEIGHTS,
        'thresholds': {
            'calm':          CALM_THRESHOLD,
            'energy':        ENERGY_THRESHOLD,
            'valence_floor': VALENCE_FLOOR,
        },
        'pre_filters': {
            'speechiness_max': SPEECHINESS_MAX,
            'liveness_max':    LIVENESS_MAX,
        },
        'class_distribution': {
            cls: int(counts.get(cls, 0)) for cls in ['calm', 'energy', 'other']
        },
    }
    config_path = MODELS_DIR / f'{codename}_config.json'
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=2)
    print(f'  {codename}: config -> {config_path.name}')

print('\nKlaar.')

---

## 5. Diagnostiek

### 5a. Klasseverdeling (tabel)

Een grote OTHER-klasse is normaal -- die is bedoeld als opvangnet voor muziek
die niet duidelijk calm of energetisch is. CALM en ENERGY zijn de relevante pools
voor playlistgeneratie.

In [ ]:
print(f"{'Deelnemer':<14} {'CALM':>8} {'ENERGY':>8} {'OTHER':>8} {'Totaal':>8}")
print('-' * 42)
for codename, (df_c, _, _) in results.items():
    counts = df_c['class'].value_counts()
    n_calm   = counts.get('calm', 0)
    n_energy = counts.get('energy', 0)
    n_other  = counts.get('other', 0)
    total    = len(df_c)
    print(f'{codename:<14} {n_calm:>7} {n_energy:>8} {n_other:>8} {total:>8}')
    for cls, n in [('calm', n_calm), ('energy', n_energy)]:
        if n / total < 0.05:
            print(f'  Let op: {cls.upper()} < 5% -- overweeg drempelwaarden aan te passen')

### 5b. Klasseverdeling (gestapeld staafdiagram)

In [ ]:
fig, ax = plt.subplots(figsize=(max(6, len(participants) * 2), 5))

x      = np.arange(len(participants))
bottom = np.zeros(len(participants))

for cls in CLASS_ORDER:
    counts = [(results[p][0]['class'] == cls).sum() for p in participants]
    bars = ax.bar(x, counts, bottom=bottom, color=CLASS_COLORS[cls],
                  label=cls.upper(), width=0.55, edgecolor='none')
    totals = [len(results[p][0]) for p in participants]
    for i, (bar, count, total) in enumerate(zip(bars, counts, totals)):
        pct = count / total * 100
        if pct > 4:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bottom[i] + count / 2,
                f'{pct:.0f}%',
                ha='center', va='center', fontsize=8, color='#0f1218', fontweight='bold',
            )
    bottom += np.array(counts, dtype=float)

ax.set_xticks(x)
ax.set_xticklabels(participants)
ax.set_ylabel('Aantal nummers')
ax.set_title('Klasseverdeling per deelnemer')
ax.legend(loc='upper right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 5c. Gemiddelde kenmerken per klasse

In [ ]:
for codename, (df_c, _, _) in results.items():
    print(f"\n{'='*65}")
    print(f'  {codename} -- gemiddelde kenmerken per klasse')
    print(f"{'='*65}")
    means = (
        df_c.groupby('class')[SCORING_FEATURES + ['arousal_score']]
        .mean()
        .round(3)
        .reindex(['calm', 'energy', 'other'])
    )
    display(means)

### 5d. Arousal-score verdeling

Een goede classificatie geeft duidelijk gescheiden pieken voor CALM (links) en ENERGY (rechts).
Grote overlap tussen CALM en OTHER is normaal -- de valence-vloer heeft er al voor gefilterd.

In [ ]:
n_cols = min(2, len(results))
n_rows = (len(results) + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows), squeeze=False)

for idx, (codename, (df_c, _, _)) in enumerate(results.items()):
    ax = axes[idx // n_cols][idx % n_cols]
    for cls in CLASS_ORDER:
        subset = df_c[df_c['class'] == cls]['arousal_score']
        ax.hist(subset, bins=40, color=CLASS_COLORS[cls], alpha=0.7,
                label=cls.upper(), edgecolor='none')
    ax.axvline(CALM_THRESHOLD, color='#56b4e9', linestyle='--',
               linewidth=1.5, label=f'Calm < {CALM_THRESHOLD}')
    ax.axvline(ENERGY_THRESHOLD, color='#e69f00', linestyle='--',
               linewidth=1.5, label=f'Energy > {ENERGY_THRESHOLD}')
    ax.set_title(codename)
    ax.set_xlabel('Arousal-score')
    ax.set_ylabel('Aantal nummers')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for idx in range(len(results), n_rows * n_cols):
    axes[idx // n_cols][idx % n_cols].set_visible(False)

fig.suptitle('Arousal-score verdeling per deelnemer', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 5e. Drempelwaarde-gevoeligheidsanalyse

Sweept calm- en energy-drempels om te zien hoe de klasseverdeling verschuift.
Helpt bij het vinden van stabiele drempelgebieden die robuust zijn voor kleine parameterwijzigingen.

In [ ]:
calm_range   = np.arange(0.25, 0.46, 0.05)
energy_range = np.arange(0.55, 0.76, 0.05)

for codename, (df_c, df_scaled, _) in results.items():
    sweep_rows = []
    for ct in calm_range:
        for et in energy_range:
            if ct >= et:
                continue
            labels = [
                'calm'   if a < ct and v >= VALENCE_FLOOR else
                'energy' if a > et else 'other'
                for a, v in zip(df_c['arousal_score'], df_scaled['valence'])
            ]
            n_total = len(labels)
            sweep_rows.append({
                'calm_thresh':   round(ct, 2),
                'energy_thresh': round(et, 2),
                'n_calm':        labels.count('calm'),
                'pct_calm':      round(labels.count('calm') / n_total * 100, 1),
                'n_energy':      labels.count('energy'),
                'pct_energy':    round(labels.count('energy') / n_total * 100, 1),
                'n_other':       labels.count('other'),
            })

    sweep_df = pd.DataFrame(sweep_rows)
    print(f'\n{codename} -- drempelwaarde-sweep (top 10 op n_calm)')
    display(sweep_df.sort_values('n_calm', ascending=False).head(10).reset_index(drop=True))

    current = sweep_df[
        (sweep_df['calm_thresh']   == round(CALM_THRESHOLD, 2)) &
        (sweep_df['energy_thresh'] == round(ENERGY_THRESHOLD, 2))
    ]
    if not current.empty:
        print(f'  Huidige instellingen (calm={CALM_THRESHOLD}, energy={ENERGY_THRESHOLD}):')
        display(current.reset_index(drop=True))

### 5f. Spot-check -- 10 willekeurige nummers per klasse

Meest directe validatie: kloppen de nummers qua gevoel met hun klasse?
CALM-nummers moeten rustig en akoestisch aanvoelen; ENERGY-nummers snel en luid.

In [ ]:
spot_cols_base = ['name', 'artists', 'tempo', 'energy', 'valence', 'loudness', 'arousal_score', 'class']

for codename, (df_c, _, _) in results.items():
    spot_cols = [c for c in spot_cols_base if c in df_c.columns]
    print(f"\n{'='*70}")
    print(f'  {codename} -- spot-check')
    print(f"{'='*70}")
    for cls in ['calm', 'energy', 'other']:
        subset = df_c[df_c['class'] == cls]
        n_show = min(10, len(subset))
        print(f'\n  {cls.upper()} ({n_show} van {len(subset)} nummers)')
        display(subset[spot_cols].sample(n_show, random_state=RANDOM_STATE).reset_index(drop=True))

---

## 6. Visualisaties

### 6a. Radar chart -- gemiddeld kenmerkprofiel per klasse

Wat maakt een CALM-nummer anders dan een ENERGY-nummer?
De radar toont gemiddelde genormaliseerde audio-kenmerken per klasse.
Een goede classificatie geeft duidelijk verschillende profielen:
CALM scoort hoog op acousticness, ENERGY hoog op tempo en energy.

In [ ]:
RADAR_FEATURES = ['tempo', 'energy', 'loudness', 'valence', 'danceability', 'acousticness']

# Normaliseer naar [0,1] over alle deelnemers samen voor vergelijkbaarheid
all_songs = pd.concat([df for df, _, _ in results.values()], ignore_index=True)
feat_min  = all_songs[RADAR_FEATURES].min()
feat_max  = all_songs[RADAR_FEATURES].max()


def normalize_radar(df_sub: pd.DataFrame) -> pd.DataFrame:
    return (df_sub[RADAR_FEATURES] - feat_min) / (feat_max - feat_min + 1e-9)


n_feat  = len(RADAR_FEATURES)
angles  = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]

n_cols = min(2, len(participants))
n_rows = math.ceil(len(participants) / n_cols)
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows),
    subplot_kw={'projection': 'polar'}, squeeze=False,
)

for idx, codename in enumerate(participants):
    df_c = results[codename][0]
    ax = axes[idx // n_cols][idx % n_cols]
    ax.set_facecolor('#181e2a')

    for cls in CLASS_ORDER:
        subset = df_c[df_c['class'] == cls]
        if len(subset) == 0:
            continue
        means  = normalize_radar(subset).mean().tolist()
        values = means + means[:1]
        ax.plot(angles, values, color=CLASS_COLORS[cls], linewidth=2, label=cls.upper())
        ax.fill(angles, values, color=CLASS_COLORS[cls], alpha=0.12)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(RADAR_FEATURES, fontsize=8)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['0.25', '0.50', '0.75'], fontsize=6, color='#4a5568')
    ax.tick_params(colors='#e2e8f0')
    ax.set_title(codename, pad=14)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=7)
    ax.grid(color='#2d3748', alpha=0.5)

for idx in range(len(participants), n_rows * n_cols):
    axes[idx // n_cols][idx % n_cols].set_visible(False)

fig.suptitle('Gemiddeld kenmerkprofiel per klasse (genormaliseerd)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 6b. Feature boxplots per klasse

Detailweergave van de vier belangrijkste kenmerken.
De spreiding binnen een klasse laat zien hoe homogeen de groep is --
een smalle box betekent consistente nummers; een brede box betekent grote variatie.

In [ ]:
BOX_FEATURES = ['tempo', 'energy', 'valence', 'acousticness']

for codename in participants:
    df_c = results[codename][0]

    fig, axes = plt.subplots(1, len(BOX_FEATURES), figsize=(13, 4))

    for ax, feat in zip(axes, BOX_FEATURES):
        groups = [df_c[df_c['class'] == cls][feat].dropna() for cls in CLASS_ORDER]
        bp = ax.boxplot(
            groups,
            patch_artist=True,
            medianprops={'color': '#e2e8f0', 'linewidth': 1.5},
            whiskerprops={'color': '#4a5568'},
            capprops={'color': '#4a5568'},
            flierprops={'marker': '.', 'markerfacecolor': '#4a5568', 'markersize': 3, 'alpha': 0.5},
        )
        for patch, cls in zip(bp['boxes'], CLASS_ORDER):
            patch.set_facecolor(CLASS_COLORS[cls])
            patch.set_alpha(0.7)

        ax.set_xticks([1, 2, 3])
        ax.set_xticklabels([c.upper() for c in CLASS_ORDER], fontsize=8)
        ax.set_title(feat)
        ax.grid(True, axis='y', alpha=0.3)

    fig.suptitle(f'{codename} -- kenmerken per klasse', fontsize=12)
    plt.tight_layout()
    plt.show()